# Session 6: ORDER BY + Sorting

This interactive notebook explores single-column and multi-column sorting mechanisms in SQL: ascending vs descending ordering, tie-breaking hierarchies, alphabetic collation, and pagination using `LIMIT`.

---

## Overview of Tasks
- **Task 1:** Sort `products` table by `price ASC` (Flipkart lowest to highest price).
- **Task 2:** Extract top 5 most expensive products using `ORDER BY price DESC LIMIT 5`.
- **Task 3:** Multi-column sort `movies` table by `release_year DESC`, then `rating DESC`.
- **Task 4:** Alphabetical directory sorting (`restaurants` A-Z listing) with `LIMIT 10`.
- **Task 5:** Multi-column tie-breaker: Top 3 trending `songs` by `play_count DESC`, with ties resolved by `added_date DESC`.

### Task 1: Sort Products by Price Ascending (Lowest to Highest)

In [ ]:
import sqlite3
import os
import pandas as pd

os.makedirs("database", exist_ok=True)
db_path = os.path.join("database", "sorting_db.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    price DECIMAL(10, 2) NOT NULL
);
""")

sample_products = [
    ("USB-C Fast Charging Cable", "Accessories", 299.00),
    ("Wireless Optical Mouse", "Accessories", 599.00),
    ("Boat Airdopes 141", "Audio", 1299.00),
    ("Noise ColorFit Smartwatch", "Wearables", 1999.00),
    ("Sony WH-1000XM5 Headphones", "Audio", 26990.00),
    ("Samsung Galaxy S24 Ultra", "Smartphones", 129999.00),
    ("Apple iPhone 15 Pro", "Smartphones", 134900.00),
    ("Dell XPS 15 Laptop", "Laptops", 189990.00),
    ("Logitech Mechanical Keyboard", "Accessories", 6499.00),
    ("iPad Air M2", "Tablets", 59900.00)
]
cursor.executemany("INSERT INTO products (name, category, price) VALUES (?, ?, ?);", sample_products)
conn.commit()

df_t1 = pd.read_sql_query("SELECT id, name, category, price FROM products ORDER BY price ASC;", conn)
display(df_t1)

### Task 2: Top 5 Most Expensive Products (DESC + LIMIT 5)

In [ ]:
df_t2 = pd.read_sql_query("SELECT id, name, category, price FROM products ORDER BY price DESC LIMIT 5;", conn)
display(df_t2)

### Task 3: Movies Multi-Column Sort (Year DESC, Rating DESC)
Sorts primarily by `release_year` in descending order (latest first). Whenever two movies were released in the same year, the tie is broken by sorting their `rating` in descending order (highest rated first).

In [ ]:
cursor.execute("DROP TABLE IF EXISTS movies;")
cursor.execute("""
CREATE TABLE movies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title VARCHAR(100) NOT NULL,
    release_year INT NOT NULL,
    rating DECIMAL(3, 1) NOT NULL
);
""")

sample_movies = [
    ("Dune: Part Two", 2024, 8.6),
    ("Deadpool & Wolverine", 2024, 7.9),
    ("Civil War", 2024, 7.1),
    ("Oppenheimer", 2023, 8.9),
    ("Poor Things", 2023, 7.9),
    ("Spider-Man: Across the Spider-Verse", 2023, 8.7),
    ("Top Gun: Maverick", 2022, 8.3),
    ("The Batman", 2022, 7.8),
    ("Interstellar", 2014, 8.7),
    ("Whiplash", 2014, 8.5)
]
cursor.executemany("INSERT INTO movies (title, release_year, rating) VALUES (?, ?, ?);", sample_movies)
conn.commit()

query_t3 = "SELECT title, release_year, rating FROM movies ORDER BY release_year DESC, rating DESC;"
df_t3 = pd.read_sql_query(query_t3, conn)
display(df_t3)

### Task 4: Alphabetical Restaurant Listing (Zomato A-Z Directory)

In [ ]:
cursor.execute("DROP TABLE IF EXISTS restaurants;")
cursor.execute("""
CREATE TABLE restaurants (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(50) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    city VARCHAR(50) NOT NULL
);
""")

sample_restaurants = [
    ("Anand Sweets", "Sweets & Snacks", 4.3, "Bengaluru"),
    ("Bademiya", "Mughlai / Kebabs", 4.1, "Mumbai"),
    ("Chai Point", "Beverages", 4.0, "Delhi"),
    ("Dindigul Thalappakatti", "Biryani", 4.4, "Chennai"),
    ("Empire Restaurant", "North Indian", 4.2, "Bengaluru"),
    ("Farzi Cafe", "Modern Indian", 4.5, "Delhi"),
    ("Glen's Bakehouse", "Bakery / Desserts", 4.6, "Bengaluru"),
    ("Haldiram's", "North Indian / Sweets", 4.3, "Delhi"),
    ("Indian Coffee House", "Cafe", 4.1, "Kolkata"),
    ("Jaffer Bhai's Delhi Darbar", "Mughlai", 4.4, "Mumbai"),
    ("Karim's", "Mughlai", 4.5, "Delhi"),
    ("La Pino'z Pizza", "Italian", 4.3, "Ahmedabad"),
    ("Mavalli Tiffin Room (MTR)", "South Indian", 4.7, "Bengaluru")
]
cursor.executemany("INSERT INTO restaurants (name, cuisine, rating, city) VALUES (?, ?, ?, ?);", sample_restaurants)
conn.commit()

query_t4 = "SELECT id, name, cuisine, rating, city FROM restaurants ORDER BY name ASC LIMIT 10;"
df_t4 = pd.read_sql_query(query_t4, conn)
display(df_t4)

### Task 5: Top 3 Trending Songs with Date-Based Tie-Break
Songs are sorted by `play_count DESC`. If two tracks have identical play counts (such as *Blinding Lights* and *Starboy* at 5,000,000 plays), the more recently added song comes first (`added_date DESC`).

In [ ]:
cursor.execute("DROP TABLE IF EXISTS songs;")
cursor.execute("""
CREATE TABLE songs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title VARCHAR(100) NOT NULL,
    artist VARCHAR(100) NOT NULL,
    play_count INT NOT NULL,
    added_date DATE NOT NULL
);
""")

sample_songs = [
    ("Starboy", "The Weeknd", 5000000, "2026-09-01"),
    ("Blinding Lights", "The Weeknd", 5000000, "2026-10-15"),
    ("Tum Hi Ho", "Arijit Singh", 4200000, "2026-08-10"),
    ("Brown Munde", "AP Dhillon", 3900000, "2026-07-20"),
    ("Espresso", "Sabrina Carpenter", 6500000, "2026-10-01")
]
cursor.executemany("INSERT INTO songs (title, artist, play_count, added_date) VALUES (?, ?, ?, ?);", sample_songs)
conn.commit()

query_t5 = """
SELECT id, title, artist, play_count, added_date
FROM songs
ORDER BY play_count DESC, added_date DESC
LIMIT 3;
"""
df_t5 = pd.read_sql_query(query_t5, conn)
display(df_t5)

conn.close()